# Python coding problems

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/python/practice/python-coding-problems.ipynb)

Practice Python for data pipelines: aggregation, sorting, joins, file processing, schema changes, sessions and retries. Each problem gives the task, sample data and expected output. Try it in a scratch cell, then expand the solution to review the approach.

## Problem 1 - Deduplicate, count, and group without pandas

**Statement.** Implement the workhorses of in-memory aggregation with the standard library only:

1. `dedupe_preserving_order(records, key)` - keep the **first** occurrence of each key, in input order.
2. `frequency_map(items)` - the equivalent of `COUNT(*) GROUP BY item`.
3. `group_by(records, key)` - bucket rows per key, preserving first-seen group order.
4. `aggregate(rows)` - SQL-style `COUNT`, `COUNT(DISTINCT)`, `SUM`, and `AVG` per group.

**Example.**

```text
records = [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"a","v":3}, {"id":"c","v":4}]
dedupe   -> [{"id":"a","v":1}, {"id":"b","v":2}, {"id":"c","v":4}]
frequency_map(["us","eu","us"]) -> {"us": 2, "eu": 1}
aggregate([("us",10.0), ("us",30.0), ("us",None)])
         -> {"us": {"rows": 3, "distinct": 2, "total": 40.0, "avg": 20.0}}
```

**Constraints.**

- O(n) time; no pandas; do not mutate the input records.
- Preserve input order for dedupe, and first-seen order for groups.
- Decide and document null behaviour: here a null *key* is dropped, while a null *value* counts as a row but is excluded from `SUM`, `AVG` and `COUNT(DISTINCT)`. This exercise returns a total of `0.0` and an average of `None` for an all-null group.

<details>
<summary><b>Show solution and explanation</b></summary>

### 1a. Dedupe preserving order

The whole problem is one pass with a `set` of keys already seen. A set gives average O(1) membership, so the pass is O(n) time and O(k) memory for k distinct keys -- the same shape as a Spark `dropDuplicates`, but with deterministic first-seen order rather than arbitrary order.

The pitfall is null handling: a `None` key is not a real business key (it would merge unrelated rows), so it is dropped rather than treated as one group. Note also that the records themselves are appended by reference and never copied or mutated.

```python
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def dedupe_preserving_order(records: Iterable[dict[str, Any]], key: str = "id") -> list[dict[str, Any]]:
    """Keep the first record seen for each value of `key`, in input order."""
    seen: set[Any] = set()
    out: list[dict[str, Any]] = []
    for record in records:
        value = record.get(key)
        if value is None or value in seen:
            continue
        seen.add(value)
        out.append(record)
    return out
```

### 1b. `frequency_map` - `COUNT(*) GROUP BY item`

`collections.Counter` is a dict subclass specialised for counting, so the implementation is a one-liner that is still O(n) time and O(k) memory. Because `Counter` iterates in first-seen order, `most_common` and the raw dict both produce deterministic output -- important when a test or a dashboard compares two runs.

Keep the input out of the pipeline after this: `frequency_map` doubles as the reducer for a distributed word count once each partition has emitted its local counts.

```python
def frequency_map(items: Iterable[Any]) -> dict[Any, int]:
    """COUNT(*) GROUP BY item, without pandas."""
    return dict(Counter(items))
```

### 1c. `group_by` - bucket rows per key

A `defaultdict(list)` plus one append per record is the in-memory `GROUP BY`. Because dicts preserve insertion order since Python 3.7, the resulting groups come out in first-seen order, not sorted -- a detail worth stating in an interview, since a downstream `ORDER BY` is then unnecessary.

The method retains the original row objects, so the memory cost is O(n), not O(k). Mutating a row inside a group mutates the caller's data.

```python
def group_by(records: Iterable[dict[str, Any]], key: str) -> dict[Any, list[dict[str, Any]]]:
    """GROUP BY key -> list of rows, preserving first-seen group order."""
    groups: dict[Any, list[dict[str, Any]]] = defaultdict(list)
    for record in records:
        groups[record.get(key)].append(record)
    return dict(groups)
```

### 1d. `aggregate` - `COUNT`, `COUNT(DISTINCT)`, `SUM`, `AVG`

One accumulator dict per group holds the row count, a set for distinct values, and the running total. A `None` value increments `rows` (so `COUNT(*)` matches SQL) but is skipped for the distinct set and the total, which is exactly `COUNT(col)`/`SUM(col)` semantics.

The average is computed once at the end from the total, never by averaging averages. That is the single most common aggregation bug in pipelines, and it is why `AVG` is not additive across partitions.

```python
def aggregate(rows: Iterable[tuple[Any, float | None]]) -> dict[Any, dict[str, float]]:
    """COUNT / COUNT(DISTINCT) / SUM / AVG grouped by the first tuple element."""
    buckets: dict[Any, dict[str, Any]] = defaultdict(lambda: {"rows": 0, "non_null": 0, "seen": set(), "total": 0.0})
    for group, value in rows:
        bucket = buckets[group]
        bucket["rows"] += 1
        if value is not None:
            bucket["non_null"] += 1
            bucket["seen"].add(value)
            bucket["total"] += value
    return {
        group: {
            "rows": bucket["rows"],
            "distinct": len(bucket["seen"]),
            "total": bucket["total"],
            "avg": bucket["total"] / bucket["non_null"] if bucket["non_null"] else None,
        }
        for group, bucket in buckets.items()
    }
```

</details>

**Complexity & edge cases.** Dedupe and counting are O(n) time; dedupe uses O(k) memory for k distinct keys, while `group_by`/`aggregate` use O(n) because they retain rows or distinct values. Edge cases: empty input returns `[]`/`{}`; unhashable keys must be normalised to tuples; `1 == True` and `1 == 1.0` collide in a Python dict, so normalise types on ingest; summing money in `float` drifts, so accumulate integer cents or `Decimal`; keep order in a list because iterating a `set` is not order-stable across runs.

**Interview follow-ups.**

1. *The data no longer fits in RAM - how does each operation change?* Sort-merge or hash-partition on disk; approximate distinct counts with HyperLogLog; a group-by becomes a shuffle in Spark.
2. *Why is `dict`/`set` lookup called O(1), and when is it not?* Average-case hashing; worst case O(n) with adversarial collisions; keys must be hashable with a stable hash.

## Problem 2 - Multi-key sorting with key functions

**Statement.** Reproduce `ORDER BY status ASC, amount DESC, id ASC`, then generalise to any number of `(field, descending)` pairs. Also handle nulls-last and the case where a key function cannot express the comparison.

**Example.**

```text
input:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"b","amount":2,"id":2}, {"status":"a","amount":1,"id":4}]
output:
  [{"status":"a","amount":3,"id":3}, {"status":"a","amount":1,"id":1},
   {"status":"a","amount":1,"id":4}, {"status":"b","amount":2,"id":2}]
```

**Constraints.** Use a key function where possible; avoid `cmp_to_key` in the hot path; `None` must sort last and never raise `TypeError`; do not mutate the input list.

<details>
<summary><b>Show solution and explanation</b></summary>

### 2a. One tuple key for a fixed `ORDER BY`

A tuple key is compared element by element, so `(status, -amount, id)` reproduces `status ASC, amount DESC, id ASC` in a single pass. Negating a numeric field is the cheapest way to flip its direction, and it works because the key is computed once per row (decorate-sort-undecorate) rather than once per comparison.

The catch is that `-x` only exists for numbers. Strings, dates, and custom objects cannot be negated, which is what forces the general version next.

```python
from functools import cmp_to_key
from typing import Any


def sort_orders(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """ORDER BY status ASC, amount DESC, id ASC using a single tuple key."""
    return sorted(rows, key=lambda row: (row["status"], -row["amount"], row["id"]))
```

### 2b. Any number of mixed-direction keys

Python's Timsort is stable, so sorting repeatedly from the least-significant key to the most-significant one gives the same result as a single combined key. Each pass only has to handle one direction, so strings and dates work fine and no value is ever negated.

The cost is one extra pass per key and a copy of the list; for a handful of keys that is irrelevant next to the O(n log n) comparisons. Copy with `list(rows)` first so the caller's list is never reordered.

```python
def sort_by_keys_stable(rows: list[dict[str, Any]], keys: list[tuple[str, bool]]) -> list[dict[str, Any]]:
    """ORDER BY any number of (field, descending) pairs using stable passes."""
    out = list(rows)
    for field, descending in reversed(keys):
        out.sort(key=lambda row: row[field], reverse=descending)
    return out
```

### 2c. Nulls last

Comparing `None` with an `int` raises `TypeError`, so a null flag must lead the tuple: `False` sorts before `True`, pushing every null to the end regardless of the value type. This is the standard `NULLS LAST` emulation and it works for any comparable column.

If the direction is descending, you place the flag accordingly (`row[field] is not None`) -- a common off-by-one in interview answers.

```python
def sort_nulls_last(rows: list[dict[str, Any]], field: str) -> list[dict[str, Any]]:
    """None sorts after every real value because the boolean flag leads the key."""
    return sorted(rows, key=lambda row: (row[field] is None, row[field]))
```

### 2d. The comparator escape hatch

Sometimes the ordering is genuinely not expressible as a key -- for example custom business logic that compares two records field by field with early exit. `functools.cmp_to_key` wraps a `compare(a, b) -> -1/0/1` function into something `sorted` accepts.

It is the slow path: the comparator runs from Python on every comparison, so O(n log n) comparisons each pay interpreter overhead instead of one cheap key per element. Mention it as a fallback, not as the default.

```python
def sort_with_comparator(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    """The escape hatch when a key function cannot express the comparison."""

    def compare(left: dict[str, Any], right: dict[str, Any]) -> int:
        if left["status"] != right["status"]:
            return -1 if left["status"] < right["status"] else 1
        if left["amount"] != right["amount"]:
            return -1 if left["amount"] > right["amount"] else 1
        return (left["id"] > right["id"]) - (left["id"] < right["id"])

    return sorted(rows, key=cmp_to_key(compare))
```

</details>

**Complexity & edge cases.** Sorting is O(n log n) comparisons plus O(n) memory for `sorted` (`list.sort` is in place). Python's Timsort is stable, which is what makes multi-pass sorting and deterministic tie-breaking work. Edge cases: `None` inside a tuple key raises `TypeError` unless a null flag leads the tuple; negating fails for strings, dates, and custom objects, so prefer `reverse=` passes; naive and aware datetimes cannot be compared; `cmp_to_key` runs Python for every comparison and is several times slower than a key function.

**Interview follow-ups.**

1. *How would you sort 100 GB on one machine?* Sort each chunk, write it to disk, then k-way merge with `heapq.merge`; keep only one buffer per chunk in memory.
2. *Why does `sorted` need the key function to be cheap?* The key is computed once per element (decorate-sort-undecorate), but an expensive or non-deterministic key breaks that assumption; the comparator variant recomputes logic on every comparison.

## Problem 3 - Top-K and top-N-per-group with `heapq`

**Statement.** Return the K most frequent keys from a stream, and the largest N rows per group, without sorting everything. A full sort is O(n log n); a bounded heap is O(n log k) and stores only k elements.

**Example.**

```text
keys = ["a","b","a","c","a","b","d"], k = 2
top_k -> [("a", 3), ("b", 2)]

rows grouped by dept, top 2 salaries per dept, computed in one pass
eng: [("bob",150), ("ana",120)]
ops: [("ed",110),  ("di",100)]
```

**Constraints.** Return `[]` for `k <= 0`; ties must be deterministic; keep memory bounded per group; do not sort the full input.

<details>
<summary><b>Show solution and explanation</b></summary>

### 3a. `top_k` with `Counter.most_common`

`Counter.most_common(k)` counts everything in O(n), then uses a heap to pick the k largest counts in O(m log k) for m distinct keys. It is the shortest correct answer, and for `k == 0` it returns an empty list rather than raising.

Counting still needs O(m) memory, so for a genuinely unbounded key space this is a sketch problem (Count-Min or Space-Saving), not a `Counter` problem.

```python
import heapq
from collections import Counter, defaultdict
from collections.abc import Iterable
from typing import Any


def top_k(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """K most common keys; Counter.most_common uses a heap internally."""
    return Counter(keys).most_common(k)
```

### 3b. Explicit bounded min-heap

Building the heap by hand shows the mechanism an interviewer wants: keep a *min*-heap of size k, push while it is not full, and only replace the root when the incoming count beats the smallest kept count. Memory is O(k) beyond the counter, and the final `sorted` is O(k log k).

The tie-break `(-count, str(key))` makes equal counts deterministic; without it, two runs could return different keys.

```python
def top_k_bounded(keys: Iterable[Any], k: int) -> list[tuple[Any, int]]:
    """Explicit bounded min-heap: O(n log k) time, O(k) extra memory."""
    if k <= 0:
        return []
    heap: list[tuple[int, Any]] = []
    for key, count in Counter(keys).items():
        if len(heap) < k:
            heapq.heappush(heap, (count, key))
        elif count > heap[0][0]:
            heapq.heapreplace(heap, (count, key))
    return sorted(((key, count) for count, key in heap), key=lambda item: (-item[1], str(item[0])))
```

### 3c. Top-N per group in one pass

Keep one bounded heap per group key. Each row is pushed with its running index, so the heap orders by `(metric, index)`; popping from a min-heap drops the smallest metric and, on a tie, the earliest index. That yields a deterministic "largest N, ties broken by input order".

Memory is O(g x N) for g groups rather than O(n), which is the whole point: a per-department leaderboard should not materialise every row.

```python
def top_n_per_group(
    rows: Iterable[dict[str, Any]], group: str, metric: str, n: int
) -> dict[Any, list[dict[str, Any]]]:
    """Largest n rows per group in one pass, with bounded memory per group."""
    if n <= 0:
        return {}
    heaps: dict[Any, list[tuple[float, int, dict[str, Any]]]] = defaultdict(list)
    for index, row in enumerate(rows):
        heap = heaps[row[group]]
        # The running index breaks metric ties deterministically.
        heapq.heappush(heap, (row[metric], index, row))
        if len(heap) > n:
            heapq.heappop(heap)
    return {
        group_key: [
            row for _value, _index, row in sorted(heap, key=lambda item: (-item[0], item[1]))
        ]
        for group_key, heap in heaps.items()
    }
```

</details>

**Complexity & edge cases.** `heapq.nlargest(k, ...)`/`nsmallest` and `Counter.most_common(k)` are O(n log k) with O(k) memory. `top_n_per_group` is O(n log n) worst case but only O(g x n) memory for g groups. Edge cases: `k` larger than the number of distinct keys returns everything; ties need an explicit secondary key or the output is order-dependent; one heavy hitter dominates a `Counter`, so a Count-Min Sketch is the scalable answer; unhashable keys must be normalised first; `k <= 0` must not crash.

**Interview follow-ups.**

1. *When is `nlargest` worse than a full sort?* When k is close to n: O(n log k) approaches O(n log n) while adding heap overhead, so `sorted(...)[:k]` can win.
2. *How do you get the top K globally across partitions?* Each partition emits its local top K, then a single reducer merges K x partitions candidates with a bounded heap.

## Problem 4 - Rolling windows: max sum, moving average, percentile

**Statement.** Implement the windowed primitives behind dashboards, rate limiters, and SLOs:

1. `moving_average(values, window)` - trailing simple moving average, one value per full window.
2. `max_window_sum(values, window)` - largest sum of any contiguous window of exactly `window` elements.
3. `percentile(values, pct)` and `rolling_percentile(values, window, pct)` - linear-interpolation percentiles.
4. `trailing_event_count(timestamps, window)` - events in `(t - window, t]`, amortised O(1) per event.

**Example.**

```text
moving_average([1,2,3,4,5], 3)          -> [2.0, 3.0, 4.0]
max_window_sum([1,2,3,4,5], 3)          -> 12
percentile([10,12,13,15,20,100], 50)    -> 14.0
trailing_event_count([0,1,2,10,11], 3)  -> [1, 2, 3, 1, 2]
```

**Constraints.** O(1) update per element for the average and the count; document the window boundary; raise on a non-positive window; handle a window larger than the input.

<details>
<summary><b>Show solution and explanation</b></summary>

### 4a. Moving average - sliding sum, not a re-sum

Add the incoming value and subtract the one leaving the window, so each step is O(1) instead of O(window). The output has exactly `len(values) - window + 1` entries: one value per *full* window, with the partial prefix discarded.

Because the running total is a float, long windows accumulate rounding error; `math.fsum` on an occasional reset, or Kahan summation, keeps it honest. A window larger than the input returns `[]` rather than a partial average.

```python
import math
from collections import deque
from collections.abc import Sequence


def moving_average(values: Sequence[float], window: int) -> list[float]:
    """Trailing simple moving average, one output per full window."""
    if window <= 0:
        raise ValueError("window must be positive")
    if window > len(values):
        return []
    out: list[float] = []
    total = 0.0
    for index, value in enumerate(values):
        total += value
        if index >= window:
            total -= values[index - window]
        if index >= window - 1:
            out.append(total / window)
    return out
```

### 4b. Maximum window sum

The same sliding total, but keep the best value seen instead of emitting every window. This is the O(n) shape used for "busiest 5-minute bucket" style questions; the naive recompute is O(n x window).

Returning `None` (rather than `0` or `[]`) for an impossible window keeps the failure explicit. Negative values matter: initialising `best` to 0 would be wrong on an all-negative array, so seed it with the first window's sum.

```python
def max_window_sum(values: Sequence[float], window: int) -> float | None:
    """Largest sum of any contiguous window of exactly `window` elements."""
    if window <= 0 or window > len(values):
        return None
    total = sum(values[:window])
    best = total
    for index in range(window, len(values)):
        total += values[index] - values[index - window]
        best = max(best, total)
    return best
```

### 4c. Percentile with linear interpolation

Sort once, map the percentile to a fractional position `(n - 1) * pct / 100`, then interpolate between the two neighbouring order statistics. This matches the numpy/`pandas` default, so the numbers agree with a notebook the interviewer may compare against.

Two pitfalls: `[]` has no percentile (raise), and `pct` outside `[0, 100]` is a caller bug, not something to clamp silently. On a handful of points, p99 is simply the maximum.

```python
def percentile(values: Sequence[float], pct: float) -> float:
    """Linear-interpolation percentile (the numpy default); pct in [0, 100]."""
    if not values:
        raise ValueError("empty sample")
    if not 0 <= pct <= 100:
        raise ValueError("pct must be in [0, 100]")
    ordered = sorted(values)
    position = (len(ordered) - 1) * pct / 100
    lower = math.floor(position)
    upper = math.ceil(position)
    if lower == upper:
        return float(ordered[int(position)])
    weight = position - lower
    return ordered[lower] * (1 - weight) + ordered[upper] * weight
```

### 4d. Rolling percentile - the honest O(n x w log w) version

Re-sorting each window is the correct baseline to write first. It is deliberately not the fancy answer: an exact rolling quantile needs an order-statistic structure (two heaps or a balanced tree) or an approximate sketch, and saying so out loud is worth more than silently shipping a slow loop.

The slice `values[i : i + window]` builds a fresh list per window, so the memory churn is O(n x w) allocations over the run.

```python
def rolling_percentile(values: Sequence[float], window: int, pct: float) -> list[float]:
    """Percentile of each trailing window (recomputes the sort per window)."""
    if window <= 0:
        raise ValueError("window must be positive")
    return [percentile(values[i : i + window], pct) for i in range(len(values) - window + 1)]
```

### 4e. Trailing event count - the rate-limiter window

A `deque` of live timestamps gives O(1) amortised work per event: append the new timestamp, then pop from the left while the oldest event is outside the window. Each timestamp is pushed and popped at most once across the whole run.

The boundary is documented as half-open on the left: an event at exactly `t - window` is dropped, so the window is `(t - window, t]`. Choosing that convention explicitly prevents the classic off-by-one in rate limiters.

```python
def trailing_event_count(timestamps: Sequence[int], window: int) -> list[int]:
    """Deque-based count of events in (t - window, t]; amortised O(1) per event."""
    buffer: deque[int] = deque()
    out: list[int] = []
    for current in timestamps:
        buffer.append(current)
        while buffer and buffer[0] <= current - window:
            buffer.popleft()
        out.append(len(buffer))
    return out
```

</details>

**Complexity & edge cases.** The moving average and event count are O(n) time with O(window) memory (the deque only ever holds live events); `max_window_sum` is O(n) with O(1) extra space; `rolling_percentile` is O(n x w log w) because each window is re-sorted. Edge cases: a window larger than the input yields `[]`/`None`; define whether the window is inclusive at the left edge (here it is exclusive: `t - window` is dropped); empty samples raise in `percentile`; floating-point averages can drift, so use `math.fsum` or Kahan summation for very long windows; p99 on a handful of points is just the maximum and is statistical noise.

**Interview follow-ups.**

1. *How do you compute an exact rolling median efficiently?* Keep a max-heap of the lower half and a min-heap of the upper half, rebalancing on insert/evict: O(log window) per element.
2. *Why are percentiles not additive across buckets?* The p99 of a union is not the max or average of the per-bucket p99s - you must merge the underlying sketches or raw samples.

## Problem 5 - Two pointers and prefix sums

**Statement.** Find two values in a sorted array that sum to a target (two pointers), answer range-sum queries in O(1) (prefix sums), count subarrays summing to a target (prefix sums + hash map), and find the first element at or after a target (`bisect`).

**Example.**

```text
two_sum_sorted([1,3,4,6,8,11], 10)   -> (2, 3)          # values 4 + 6
build_prefix_sums([2,4,6,8])         -> [0, 2, 6, 12, 20]
range_sum(prefix, 1, 3)              -> 10               # 4 + 6
count_subarrays_with_sum([1,1,1], 2) -> 2
first_at_or_after([1,3,3,7], 3)      -> 3
```

**Constraints.** Two pointers is O(n) time and O(1) space on already-sorted input; prefix arrays build in O(n) and answer in O(1); support negative numbers.

<details>
<summary><b>Show solution and explanation</b></summary>

### 5a. Two pointers on a sorted array

Start at both ends: if the sum is too small the only way to grow it is to advance the left pointer, and if it is too large the only way to shrink it is to retreat the right pointer. Each index moves at most n times, so the scan is O(n) time and O(1) space.

The precondition is that the data is sorted; on unsorted data the move decision is invalid and you must hash instead. Returning indices (not values) keeps duplicates unambiguous, and any valid pair is acceptable.

```python
from bisect import bisect_left, bisect_right
from collections.abc import Sequence
from itertools import accumulate


def two_sum_sorted(nums: Sequence[int], target: int) -> tuple[int, int] | None:
    """Two-pointer scan on a sorted array; O(n) time, O(1) space."""
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return left, right
        if total < target:
            left += 1
        else:
            right -= 1
    return None
```

### 5b. Prefix sums

Store `prefix[i] = sum(values[:i])` with a leading zero, so the array has length `n + 1` and any range `[lo, hi)` is one subtraction: `prefix[hi] - prefix[lo]`. Building is O(n) with `itertools.accumulate`, and every later query is O(1).

The leading zero is what makes `lo = 0` work without a branch. Skipping it is the classic off-by-one that forces a special case for the first element.

```python
def build_prefix_sums(values: Sequence[int]) -> list[int]:
    """prefix[i] = sum(values[:i]); length n + 1 so prefix[0] == 0."""
    return [0, *accumulate(values)]
```

### 5c. O(1) range queries with validation

With the prefix array in hand, `range_sum` is two lookups. Validate `0 <= lo <= hi <= len(prefix) - 1` up front: an invalid range must raise rather than silently return a negative or wrapped number, because a wrong aggregate is far worse than a failed job.

An empty range (`lo == hi`) is valid and returns 0, matching the mathematical convention.

```python
def range_sum(prefix: Sequence[int], lo: int, hi: int) -> int:
    """sum(values[lo:hi]) in O(1) once the prefix array is built."""
    if not 0 <= lo <= hi <= len(prefix) - 1:
        raise ValueError("invalid range")
    return prefix[hi] - prefix[lo]
```

### 5d. Counting subarrays that sum to a target

Keep a hash map from each running prefix sum to how many times it has occurred, seeded with `{0: 1}` for the empty prefix. At every step, the number of subarrays ending here is the count of times `running - target` has been seen.

This is O(n) time and O(n) memory, and it is the version that *works with negatives*, where a sliding window does not. The `{0: 1}` seed is what counts subarrays that start at index 0.

```python
def count_subarrays_with_sum(values: Sequence[int], target: int) -> int:
    """Count contiguous subarrays summing to target (works with negatives)."""
    seen = {0: 1}
    running = 0
    count = 0
    for value in values:
        running += value
        count += seen.get(running - target, 0)
        seen[running] = seen.get(running, 0) + 1
    return count
```

### 5e. Lower and upper bound with `bisect`

`bisect_left` returns the first index at or after the target and `bisect_right - 1` the last index at or before it; both are O(log n) and run in C. Convert the index to a value and return `None` at the edges so callers do not have to catch `IndexError`.

Interviews care about the pair because it answers "first row at or after this watermark" (catch-up) and "last partition at or before this key" (lookup) without scanning. Any monotonic ordering works, which is why ISO date strings are the standard partition key.

```python
def first_at_or_after(sorted_values: Sequence[int], target: int) -> int | None:
    """Lower bound via bisect; None when every value is smaller than target."""
    index = bisect_left(sorted_values, target)
    return sorted_values[index] if index < len(sorted_values) else None


def last_at_or_before(sorted_values: Sequence[int], target: int) -> int | None:
    """Upper bound minus one; None when every value is larger than target."""
    index = bisect_right(sorted_values, target) - 1
    return sorted_values[index] if index >= 0 else None
```

</details>

**Complexity & edge cases.** Two pointers is O(n) time / O(1) space but only valid on sorted data; a prefix array builds in O(n) and each query is O(1); `count_subarrays_with_sum` is O(n) time / O(n) memory and handles negatives (sliding windows do not). Edge cases: duplicate values are fine because any valid pair is acceptable; `lo == hi` returns 0; empty input; `bisect_left` gives the first equal element, `bisect_right - 1` gives the last; a monotonic predicate is required for binary search, so date partitions must be zero-padded ISO strings.

**Interview follow-ups.**

1. *Why does a sliding window fail with negative numbers?* Shrinking the window does not monotonically reduce the sum, so you cannot decide when to move the left edge.
2. *How do prefix sums extend to 2-D?* Build a summed-area table where `S[i][j]` is the sum of the rectangle above-left of `(i, j)`; each rectangle query is four lookups.

## Problem 6 - Merge overlapping intervals

**Statement.** Merge every overlapping or touching half-open `[start, end)` interval - how a scheduler coalesces file-availability windows and how compaction groups small files.

**Example.**

```text
merge([[1,3],[2,6],[8,10],[15,18]]) -> [[1,6],[8,10],[15,18]]
merge([[1,4],[4,5]])                -> [[1,5]]     # touching merges
merge([[1,10],[2,3],[4,5]])         -> [[1,10]]    # nested absorbed
```

**Constraints.** O(n log n) time; do not mutate the input; raise on `start > end`; keep zero-length intervals; merge touching endpoints by convention (even though adjacent half-open intervals do not overlap).

<details>
<summary><b>Show solution and explanation</b></summary>

### 6a. Sort, then one sweep

Sorting by start time reduces the problem to a single left-to-right pass: each interval either extends the last merged interval (`start <= merged[-1][1]`) or begins a new one. The sort dominates at O(n log n); the sweep is O(n) time and O(n) output.

Treating touching endpoints as overlapping (`<=`) is the half-open convention: `[1, 4)` and `[4, 5)` cover a continuous span with no gap. Switching to `<` gives closed intervals instead -- pick one and document it.

```python
from collections.abc import Iterable


def merge_intervals(intervals: Iterable[tuple[int, int]]) -> list[tuple[int, int]]:
    """Merge overlapping or touching half-open [start, end) intervals."""
    merged: list[tuple[int, int]] = []
    for start, end in sorted(intervals):
        if start > end:
            raise ValueError(f"invalid interval: {(start, end)}")
        if merged and start <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], end))
        else:
            merged.append((start, end))
    return merged
```

</details>

**Complexity & edge cases.** O(n log n) for the sort plus O(n) for the sweep and the output; `sorted` copies, so the input is untouched. Edge cases: empty input returns `[]`; zero-length intervals survive; closed intervals need `start < merged[-1][1]` so touching endpoints stay separate; nested intervals are absorbed by `max`; huge coordinates favour an interval list over a bitmap; streaming intervals that arrive unordered require a buffer and can only be partially merged.

**Interview follow-ups.**

1. *How would you find the peak number of concurrent intervals?* Sweep the endpoints as `(+1, start)` and `(-1, end)` events, sort, and track a running maximum - O(n log n), or O(n) counting sort on small integer ranges.
2. *How does this relate to small-file compaction?* Group adjacent files whose size range overlaps or is below a threshold, then rewrite each group as one file - the same merge-with-tolerance idea.

## Problem 7 - Binary search over sorted partitions

**Statement.** A table is stored as sorted ISO-date partitions. (a) Find the partition that can contain a key, or the first partition at or after a watermark. (b) Generalise the partitioning idea: find the k-th smallest element of two sorted arrays by binary-searching the split point, which is the "median of two sorted arrays" interview classic.

**Example.**

```text
partitions = ["2026-01-01","2026-02-01","2026-03-01","2026-04-01"]
partition_for_key("2026-03-15")   -> "2026-03-01"
first_at_or_after("2026-03-15")   -> "2026-04-01"

kth_smallest_two_sorted([1,3,8], [2,4,5,6,7], k=4) -> 4
```

**Constraints.** O(log n) per lookup; O(log(min(n, m))) for the two-array selection; handle empty inputs and out-of-range `k`; the sorted arrays must be ascending (raise if not).

<details>
<summary><b>Show solution and explanation</b></summary>

### 7a. Partition lookup for a key

`bisect_right(partitions, key) - 1` is the last partition start that is `<=` the key: the file that could contain it. This is the prune step in partition elimination, and it is O(log n) with no scanning.

`bisect_right` (not `bisect_left`) is what makes an exact partition-boundary key resolve to its own partition rather than the previous one. A key before every partition returns `None`.

```python
from bisect import bisect_left, bisect_right
from collections.abc import Sequence


def partition_for_key(partitions: Sequence[str], key: str) -> str | None:
    """Last partition whose start is <= key (the file that can contain key)."""
    index = bisect_right(partitions, key) - 1
    return partitions[index] if index >= 0 else None
```

### 7b. Watermark resume point

`bisect_left(partitions, watermark)` is the first partition at or after the watermark, i.e. where incremental processing resumes after the last committed watermark. The two helpers are deliberately mirrored: one looks back for a lookup, one looks forward for catch-up.

The partition strings must be zero-padded ISO dates, because `bisect` compares lexicographically and `"2026-1-01"` would sort after `"2026-12-01"`.

```python
def first_at_or_after(partitions: Sequence[str], watermark: str) -> str | None:
    """First partition at or after a watermark (where incremental processing resumes)."""
    index = bisect_left(partitions, watermark)
    return partitions[index] if index < len(partitions) else None
```

### 7c. k-th smallest of two sorted arrays

Binary-search the *split point* instead of the value: take `i` elements from `a` and `j = k - i` from `b`, and check that the largest left element is no bigger than the smallest right element on both sides. Each step halves the search range, so the selection is O(log(min(n, m))).

Always search the shorter array: it bounds the range and keeps both partition indices inside their arrays. The sentinels `-inf`/`inf` remove the boundary special cases. Unsorted input must be rejected explicitly, because the invariant fails silently otherwise.

```python
def kth_smallest_two_sorted(a: Sequence[int], b: Sequence[int], k: int) -> int:
    """k-th smallest (1-indexed) of two sorted arrays via partition binary search."""
    if not 1 <= k <= len(a) + len(b):
        raise ValueError("k out of range")
    if any(a[i] > a[i + 1] for i in range(len(a) - 1)) or any(b[i] > b[i + 1] for i in range(len(b) - 1)):
        raise ValueError("inputs must be sorted")
    if len(a) > len(b):
        a, b = b, a  # binary-search the shorter array
    n, m = len(a), len(b)
    low, high = max(0, k - m), min(k, n)
    while low <= high:
        i = (low + high) // 2          # take i elements from a
        j = k - i                      # ... and j elements from b
        a_left = a[i - 1] if i > 0 else float("-inf")
        a_right = a[i] if i < n else float("inf")
        b_left = b[j - 1] if j > 0 else float("-inf")
        b_right = b[j] if j < m else float("inf")
        if a_left <= b_right and b_left <= a_right:
            return max(a_left, b_left)
        if a_left > b_right:
            high = i - 1
        else:
            low = i + 1
    raise RuntimeError("unreachable: inputs were not sorted")
```

### 7d. Median of two sorted arrays

An even total length needs the average of the two middle order statistics, an odd total just the middle one. Building `median_two_sorted` on top of the k-th selection keeps one tested primitive and makes the even/odd branch obvious.

Two empty arrays have no median, so raise; one empty array is fine and reduces to a normal index.

```python
def median_two_sorted(a: Sequence[int], b: Sequence[int]) -> float:
    """Median of two sorted arrays built on the k-th selection."""
    total = len(a) + len(b)
    if total == 0:
        raise ValueError("both arrays are empty")
    if total % 2 == 1:
        return float(kth_smallest_two_sorted(a, b, total // 2 + 1))
    lower = kth_smallest_two_sorted(a, b, total // 2)
    upper = kth_smallest_two_sorted(a, b, total // 2 + 1)
    return (lower + upper) / 2
```

</details>

**Complexity & edge cases.** `bisect` gives O(log n) lookups in C; the two-array selection is O(log(min(n, m))) time and O(1) space. Edge cases: empty partition lists; duplicate partition starts (`bisect_left` -> first, `bisect_right` -> last); lexicographic order only equals chronological order for zero-padded ISO dates; `k` out of range; one empty array; even total length needs the average of two order statistics; unsorted input must be rejected, because the partition invariant silently breaks.

**Interview follow-ups.**

1. *Why binary-search the shorter array?* It bounds the search range at min(n, m) and keeps the partition indices valid, giving the tighter O(log(min(n,m))) bound.
2. *How do you binary-search data in a data lake?* Store per-file min/max statistics (Parquet footers, Iceberg/Delta manifests) and binary-search the sorted manifest instead of scanning every file.

## Problem 8 - Generator-based streaming of a large file

**Statement.** Process a file far larger than RAM on one machine. Read it lazily, transform in bounded batches, and never hold the whole file. Generators keep memory flat.

**Example.**

```text
File events.csv contains:
user_id,amount
u1,10.5
u2,3
u1,2.5
bad-row

totals_by_user(Path("events.csv"))
Expected: {"u1": 13.0, "u2": 3.0}

first_n_matching(Path("events.csv"), "u1,", 1)
Expected: ["u1,10.5"]
```

**Constraints.** O(1) memory for the reader, independent of file size; handle plain and gzip files; skip blank lines; dead-letter malformed rows instead of aborting; no third-party libraries.

<details>
<summary><b>Show solution and explanation</b></summary>

### 8a. A lazy line reader

A generator function holds one line at a time and suspends between yields, so the open file handle is the only state. Iterating a file object already streams with a bounded buffer -- the generator just layers CRLF stripping and transparent gzip opening on top.

The `with` block closes the handle when the generator is exhausted *or* garbage-collected early. Never call `read()`/`readlines()` here; that is the single decision that makes the difference between working and OOM on a file bigger than RAM.

```python
import gzip
from collections import defaultdict
from collections.abc import Iterator
from pathlib import Path

try:  # Python 3.12+
    from itertools import batched
except ImportError:  # Python 3.11 and older, including some Colab runtimes
    from itertools import islice

    def batched(iterable, n):
        "Batch data into tuples of length n; the last batch may be shorter."
        it = iter(iterable)
        while chunk := tuple(islice(it, n)):
            yield chunk


def read_lines(path: Path) -> Iterator[str]:
    """Lazily yield stripped lines, transparently handling .gz files."""
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as handle:
        for line in handle:
            yield line.rstrip("\r\n")
```

### 8b. One streaming pass to per-user totals

Consume the lazily-read lines in fixed batches and accumulate into a dict keyed by user. Memory is O(batch size + number of distinct users), not O(file size) -- this is the standard single-pass aggregation that a map-reduce reducer performs per partition.

Malformed rows go to the dead-letter path: skip and continue rather than aborting the whole job. Note the order of operations: cast first, *then* add, because `totals[user_id] += value` on a `defaultdict` would insert `0.0` before the cast raises and leave a ghost user behind.

```python
def totals_by_user(path: Path) -> dict[str, float]:
    """One streaming pass: O(batch + distinct users) memory, not O(file)."""
    totals: dict[str, float] = defaultdict(float)
    for batch in batched(read_lines(path), 1_000):
        for line in batch:
            if not line:
                continue
            user_id, _, amount = line.partition(",")
            try:
                value = float(amount)
            except ValueError:
                continue  # dead-letter malformed rows instead of aborting
            # Add only after the cast succeeds: `totals[user_id] += ...` would
            # create a 0.0 entry via defaultdict before the cast raises.
            totals[user_id] += value
    return dict(totals)
```

### 8c. Short-circuiting search

`first_n_matching` stops reading as soon as it has `n` hits, which is the whole benefit of a generator: an early `break` closes the underlying file instead of scanning the remaining gigabytes. This is how a "find the first bad record in today's dump" tool stays fast.

The alternative -- filtering a materialised list -- pays for the full scan before returning anything.

```python
def first_n_matching(path: Path, needle: str, n: int) -> list[str]:
    """Short-circuiting search over a huge file: stop as soon as n matches exist."""
    matches: list[str] = []
    for line in read_lines(path):
        if needle in line:
            matches.append(line)
            if len(matches) == n:
                break
    return matches
```

</details>

**Complexity & edge cases.** O(n) time, O(batch size + retained state) memory; the reader holds one line at a time. Edge cases: a final line without a newline; CRLF endings; malformed records should be dead-lettered with a reason rather than killing the job; sniff gzip magic bytes instead of trusting the suffix in production; parallel workers need byte ranges that resync to the next newline; closing a generator early still runs the `with` block via `GeneratorExit`.

**Interview follow-ups.**

1. *How do you parallelise a single huge file?* Give each worker a byte range, seek, discard the partial first line, and stop at the last newline; gzip blocks can be decompressed in parallel with `zlib`-level offsets.
2. *Generator vs `itertools.batched` list chunks - what changes?* Batching gives amortised per-item overhead and lets you hand whole chunks to a vectorised or bulk-insert path, at the cost of holding one batch in memory.

## Problem 9 - Parse CSV, JSON, and NDJSON

**Statement.** Read the common landing formats without pandas: header CSV (with quoted fields and a BOM), a JSON array, NDJSON (one object per line), and a memory-flat streaming JSON array via `JSONDecoder.raw_decode`.

**Example.**

```text
parse_csv('order_id,amount,note\n1,10.5,"a, b"\n')
Expected: [{"order_id": "1", "amount": "10.5", "note": "a, b"}]

parse_json_array('[{"order_id": 1, "amount": 10.5}]')
Expected: [{"order_id": 1, "amount": 10.5}]

parse_ndjson('{"order_id": 1}\n{"order_id": 2}\n')
Expected: [{"order_id": 1}, {"order_id": 2}]
```

**Constraints.** These functions accept text already held in memory and materialize their results. Explain how accepting a file iterator would bound reader memory; report bad NDJSON with a line number; reject a non-array JSON document in `parse_json_array`; strip a UTF-8 BOM; no pandas.

<details>
<summary><b>Show solution and explanation</b></summary>

### 9a. Header CSV with `csv.DictReader`

`csv.DictReader` maps each row onto the header names and understands quoting, so `"a, b"` stays one field. Wrapping the text in `io.StringIO` keeps the function testable on a string while the same code works on an open file object.

The BOM is stripped explicitly: Excel and many exporters prepend `\ufeff`, which otherwise glues itself to the first header name and breaks every lookup on that column. In production you would open with `encoding="utf-8-sig"` rather than slicing.

```python
import csv
import io
import json
from collections.abc import Iterator
from typing import Any


def parse_csv(text: str) -> list[dict[str, str]]:
    """Header-based CSV; csv handles quoted fields that contain commas/newlines."""
    return list(csv.DictReader(io.StringIO(text.lstrip("\ufeff"))))
```

### 9b. A JSON array

`json.loads` returns a Python object, but the caller expects a list of records, so a top-level object or scalar is a contract violation and must raise. `parse_json_array` therefore materialises the whole document -- acceptable for a small config payload, not for a multi-GB export.

Deeply nested documents are where `json` recursion limits and memory both bite; that is the cue to reach for streaming.

```python
def parse_json_array(text: str) -> list[dict[str, Any]]:
    """A JSON array must be materialised unless you stream it with raw_decode."""
    payload = json.loads(text)
    if not isinstance(payload, list):
        raise ValueError("expected a JSON array")
    return payload
```

### 9c. NDJSON with line numbers

One JSON document per line is the natural landing format for streams: each line is independent, so a writer can append and a reader can parallelise or resume. Blank lines are skipped, and a malformed line is re-raised as a `ValueError` carrying its 1-based line number.

The line number is the whole point -- "bad NDJSON somewhere in 4 GB" is not actionable, "bad NDJSON at line 91,233" is.

```python
def parse_ndjson(text: str) -> list[dict[str, Any]]:
    """One JSON object per line; blank lines skipped, bad lines reported with a line number."""
    records: list[dict[str, Any]] = []
    for number, line in enumerate(text.splitlines(), start=1):
        if not line.strip():
            continue
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError as exc:
            raise ValueError(f"bad NDJSON at line {number}") from exc
    return records
```

### 9d. Streaming a JSON array with `raw_decode`

`JSONDecoder.raw_decode(text, index)` parses exactly one value and returns the index where it stopped, so you can walk a large buffer one element at a time instead of building the whole list. The loop skips the structural characters that sit between values (`[`, `]`, `,`, whitespace) and yields each decoded value.

Real code feeds the decoder from a chunked file reader and keeps a tail buffer; the point of the exercise is that the parser hands you a resumable position, which is what makes streaming possible at all.

```python
def iter_json_array(text: str) -> Iterator[Any]:
    """Stream a JSON array value-by-value with raw_decode, without building a second full list; the input text is already in memory."""
    decoder = json.JSONDecoder()
    position = 0
    length = len(text)
    while position < length:
        while position < length and text[position] in " \t\r\n,[]":
            position += 1
        if position >= length:
            break
        value, position = decoder.raw_decode(text, position)
        yield value
```

</details>

**Complexity & edge cases.** CSV and NDJSON are O(n) time / O(1) memory; a JSON array is O(n) memory because the document must exist. `raw_decode` makes the array streamable but requires you to skip separators yourself. Edge cases: a UTF-8 BOM (`utf-8-sig`); embedded newlines inside quoted CSV fields; very long NDJSON lines; empty strings that break `float`; a JSON file that is an object rather than an array; deeply nested JSON needs `ijson` for true streaming.

**Interview follow-ups.**

1. *Why is NDJSON the preferred landing format for streams?* Each line is an independent, appendable, partially readable document, so a crash mid-file loses at most one line and readers can parallelise by line ranges.
2. *How do you make parsing robust to schema drift?* Parse into `dict`, coerce types with an explicit per-column cast function, and route cast failures to a dead-letter path with the raw line and error.

## Problem 10 - Flatten nested JSON into rows

**Statement.** Turn nested documents into tabular rows: (a) `flatten_record` collapses nested objects into dotted paths and indexes arrays; (b) `explode_rows` emits one output row per array element with the parent fields repeated - the in-memory equivalent of `UNNEST` / `LATERAL VIEW explode`.

**Example.**

```text
input : {"order_id":1,"customer":{"country":"us"},
         "items":[{"sku":"a","qty":2},{"sku":"b","qty":1}]}
flat  : {"order_id":1,"customer.country":"us",
         "items[0].sku":"a","items[0].qty":2,"items[1].sku":"b","items[1].qty":1}
rows  : [{"order_id":1,"customer":{...},"sku":"a","qty":2},
         {"order_id":1,"customer":{...},"sku":"b","qty":1}]
```

**Constraints.** Preserve key order; do not mutate the input; an empty array emits one parent row (left-join semantics); a missing array field behaves like an empty array.

<details>
<summary><b>Show solution and explanation</b></summary>

### 10a. Recursive flatten with dotted paths

Walk the document depth-first and build a path string as you descend; dict keys extend the path with `sep`, list elements extend it with `[index]`, and scalars are written to the output. Insertion order is preserved because `out` is filled in traversal order, which is what lets a column-selecting reader stay stable.

A literal key that already contains the separator collides with a nested path -- production code escapes it. Deeply nested input can also hit the recursion limit, at which point an explicit stack is the safer rewrite.

```python
from collections.abc import Iterable
from typing import Any


def flatten_record(record: dict[str, Any], prefix: str = "", sep: str = ".") -> dict[str, Any]:
    """Collapse nested objects into dotted paths; arrays are indexed."""
    out: dict[str, Any] = {}
    for key, value in record.items():
        path = f"{prefix}{sep}{key}" if prefix else key
        if isinstance(value, dict):
            out.update(flatten_record(value, path, sep))
        elif isinstance(value, list):
            for index, element in enumerate(value):
                if isinstance(element, dict):
                    out.update(flatten_record(element, f"{path}[{index}]", sep))
                else:
                    out[f"{path}[{index}]"] = element
        else:
            out[path] = value
    return out
```

### 10b. Explode - one row per array element

For each parent record, copy every field except the array, then emit one row per element; when the element is a dict its keys are merged into the row, so `{"sku": ..., "qty": ...}` becomes columns. The parent fields are *repeated* on every generated row, which is exactly the row-multiplication that `UNNEST` produces.

An empty or missing array emits a single parent row, matching a left join. That choice is what keeps the parent count conserved; dropping the parent would silently lose orders with no items.

```python
def explode_rows(records: Iterable[dict[str, Any]], array_field: str) -> list[dict[str, Any]]:
    """One output row per array element, repeating the parent fields (UNNEST)."""
    rows: list[dict[str, Any]] = []
    for record in records:
        values = record.get(array_field) or []
        base = {key: value for key, value in record.items() if key != array_field}
        if not values:
            rows.append(dict(base))  # left-join semantics: keep one row
            continue
        for value in values:
            row = dict(base)
            if isinstance(value, dict):
                row.update(value)
            else:
                row[array_field] = value
            rows.append(row)
    return rows
```

</details>

**Complexity & edge cases.** Both functions are O(total nodes) time; `flatten_record` uses O(depth) recursion and `explode_rows` materialises one row per array element. Edge cases: a missing parent key; an empty array should still emit one parent row to match a left join; arrays of arrays and null array elements; a literal key containing the separator collides with a nested path, so production code should escape it; deeply nested documents can exceed the recursion limit, so an explicit stack is safer.

**Interview follow-ups.**

1. *Flatten or explode - how do you choose?* Flatten when you need a wide, one-row-per-document table; explode when downstream aggregation needs one row per child event, accepting the row multiplication.
2. *How does row multiplication affect downstream counts?* A naive `COUNT(*)` after exploding counts child records, not orders - count distinct parent ids or pre-aggregate before the explode.

## Problem 11 - In-memory hash join of two datasets

**Statement.** Join a large fact stream to a small dimension without a database. Build a hash map on the smaller side and stream the larger side through it - the same shape as a broadcast join. Support inner and left joins, duplicate build keys, and SQL null semantics.

**Example.**

```text
facts      : [("c1","o1"), ("c2","o2"), ("c9","o3")]
dimensions : [("c1","Alice"), ("c2","Bob")]
inner      : [("c1","o1","Alice"), ("c2","o2","Bob")]
left       : [..., ("c9","o3",None)]
```

**Constraints.** O(n + m) time, O(m) memory for the build side; the caller supplies the smaller dimension as the build side; duplicate build keys multiply rows; NULL keys never match; reject unknown join types.

<details>
<summary><b>Show solution and explanation</b></summary>

### 11a. Materialised inner/left hash join

Index the dimension rows by key first (a `defaultdict(list)` so duplicate keys are kept), then probe with each fact row. Hits are emitted once per matching dimension row; misses are emitted with a `None` payload only for a left join.

Two SQL semantics are encoded deliberately: a `None` build key is skipped, and a `None` probe key never matches, because `NULL = NULL` is unknown in SQL. Getting either wrong changes the row count silently.

```python
from collections import defaultdict
from collections.abc import Iterable, Iterator
from typing import Any, TypeVar

K = TypeVar("K")
V = TypeVar("V")


def hash_join(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
    null: Any = None,
) -> list[tuple[K, V, Any]]:
    """Broadcast-style join: build a hash index on the right (small) side."""
    if how not in {"inner", "left"}:
        raise ValueError("how must be 'inner' or 'left'")
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:  # SQL semantics: NULL keys never match
            index[key].append(value)
    out: list[tuple[K, V, Any]] = []
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                out.append((key, fact_value, dim_value))
        elif how == "left":
            out.append((key, fact_value, null))
    return out
```

### 11b. The same join as a generator

Yielding instead of appending keeps the fact side un-materialised: memory is the build index only, which is the actual broadcast-join property. The output is consumed once, so the caller can write it straight to a sink or feed another stage.

If the consumer needs to iterate twice, it must call the function again or wrap the result in a list -- the classic generator trap.

```python
def hash_join_streaming(
    facts: Iterable[tuple[K, V]],
    dimensions: Iterable[tuple[K, Any]],
    how: str = "inner",
) -> Iterator[tuple[K, V, Any]]:
    """Same join as a generator, so the fact side is never materialised."""
    index: dict[K, list[Any]] = defaultdict(list)
    for key, value in dimensions:
        if key is not None:
            index[key].append(value)
    for key, fact_value in facts:
        matches = index.get(key) if key is not None else None
        if matches:
            for dim_value in matches:
                yield key, fact_value, dim_value
        elif how == "left":
            yield key, fact_value, None
```

</details>

**Complexity & edge cases.** O(n + m) time and O(m) memory for the m-row build side; building on the large side silently materialises the fact table. Duplicate build keys multiply rows, which is correct SQL semantics but can explode output. Edge cases: NULL keys never match in SQL (unlike `==` in Python); `1` vs `"1"` key types silently drop rows, so normalise on ingest; a single skewed key defeats the in-memory approach and needs a sort-merge or distributed shuffle.

**Interview follow-ups.**

1. *Why does Spark broadcast the small side?* It ships one copy of the small table to every executor, turning a shuffle join into a map-side join with no network shuffle of the fact data.
2. *When is a sort-merge join better than a hash join?* When neither side fits in memory, when both inputs are already sorted, or when the data is skewed and a range partition balances better than a hash.

## Problem 12 - Schema drift and late-arriving dedupe

**Statement.** Two batch-ingestion safety checks. (a) Infer the schema per batch and diff it against a baseline, classifying columns as added, removed, or retyped. (b) Deduplicate replays and out-of-order backfills by keeping the record with the greatest `(updated_at, tiebreak)` per business key.

**Example.**

```text
baseline: {"id":"int", "amount":"float"}
current : {"id":"int", "amount":"str", "coupon":"str"}
drift   : {"added":["coupon"], "removed":[], "retyped":["amount"]}

input : [("u1",t1), ("u2",t1), ("u1",t3), ("u1",t2)]
output: u1 -> the t3 record, u2 -> the t1 record
```

**Constraints.** O(rows x columns) to infer, O(columns) to diff; O(n) time / O(k) memory for dedupe where k is the distinct key count; the tiebreak makes replays deterministic; do not mutate inputs.

<details>
<summary><b>Show solution and explanation</b></summary>

### 12a. Infer the observed schema

For every row record the Python type name per column, ignoring `None` (a null carries no type information). The result is `column -> set of observed type names`, so a column that mixes `int` and `float` is visible rather than silently coerced.

This is *observed* schema, not declared schema: it can only find drift in columns that actually appear in the sample. Diffing Parquet/Avro metadata is O(columns) and catches changes in columns with no rows.

```python
from collections.abc import Iterable, Mapping
from typing import Any


def infer_schema(rows: Iterable[Mapping[str, Any]]) -> dict[str, set[str]]:
    """Column -> set of observed Python type names (None contributes nothing)."""
    schema: dict[str, set[str]] = {}
    for row in rows:
        for column, value in row.items():
            if value is not None:
                schema.setdefault(column, set()).add(type(value).__name__)
    return schema
```

### 12b. Collapse a type set into a label

A column with no non-null value has no type and is labelled `null`; a single type keeps its name; anything else is `mixed(a|b)` in sorted order. Sorting makes the label deterministic, which matters because the drift diff compares labels as strings.

Reporting `mixed(int|float)` is usually benign widening, while a type that becomes `str` is the one that breaks readers.

```python
def describe_type(types: set[str]) -> str:
    """Collapse a set of observed types into one readable label."""
    if not types:
        return "null"
    if len(types) == 1:
        return next(iter(types))
    return "mixed(" + "|".join(sorted(types)) + ")"
```

### 12c. Diff baseline against current

Set algebra on the column names gives added (`now - base`) and removed (`base - now`); the intersection is checked for a changed type label and reported as retyped. All three lists are sorted so the output is stable and diffable in CI.

The diff is O(columns) once the schemas exist, which is why comparing declared catalog metadata is far cheaper than rescanning every row.

```python
def detect_schema_drift(
    baseline: Mapping[str, set[str]], current: Mapping[str, set[str]]
) -> dict[str, Any]:
    """Classify every column as added, removed, or retyped."""
    base, now = set(baseline), set(current)
    return {
        "added": sorted(now - base),
        "removed": sorted(base - now),
        "retyped": sorted(
            column
            for column in base & now
            if describe_type(baseline[column]) != describe_type(current[column])
        ),
    }
```

### 12d. Keep the latest record per key

One pass with a dict keyed by the business key. A record replaces the stored winner only when its `(updated_at, tiebreak)` rank is strictly greater, so the first record wins ties -- which is what makes replays deterministic.

The tiebreak (`offset`, version, or ingestion id) is not optional: if two replays carry the same timestamp, a map with no secondary key picks an arbitrary winner and the sink stops converging. Tombstones and deletes must also participate here.

```python
def dedupe_latest(
    records: Iterable[dict[str, Any]],
    key: str = "id",
    ts: str = "updated_at",
    tiebreak: str = "offset",
) -> list[dict[str, Any]]:
    """Keep the record with the greatest (timestamp, tiebreak) per key."""
    winners: dict[Any, dict[str, Any]] = {}
    for record in records:
        current = winners.get(record[key])
        rank = (record[ts], record.get(tiebreak, 0))
        if current is None or rank > (current[ts], current.get(tiebreak, 0)):
            winners[record[key]] = record
    return list(winners.values())
```

</details>

**Complexity & edge cases.** Inference is O(rows x columns); the diff is O(columns), which is why diffing declared Parquet/Avro metadata beats sampling data. Dedupe is O(n) time / O(k) memory; a warehouse `MERGE` or sort-merge spills instead. Edge cases: all-null columns have no type; int-to-float widening is usually benign while string-to-number is breaking; column-order changes are invisible to a set diff but break positional CSV readers; equal timestamps need a version or Kafka-offset tiebreak; naive and aware datetimes cannot be compared, so normalise to UTC; tombstone/delete rows must win and be applied by the writer.

**Interview follow-ups.**

1. *Where should drift detection run - in the producer or the consumer?* Both: producers validate against a contract, consumers compare the landed batch against the catalog because producers lie or get bypassed.
2. *Why is the tiebreak important for idempotency?* Without it, two replays of the same timestamp can pick different winners, so the sink is non-deterministic and reruns do not converge.

## Problem 13 - Sessionize clickstream events

**Statement.** Group ordered user events into sessions, starting a new session when the idle gap exceeds a threshold - the exercise behind every product-analytics model. Handle out-of-order input and per-user interleaving.

**Example.**

```text
events: u1 at [0, 300, 2400, 2500], u2 at [0], gap = 1800 seconds
output: [u1 0-300 (2 events), u1 2400-2500 (2 events), u2 0-0 (1 event)]
```

**Constraints.** Sort by `(user, timestamp)` first; O(n log n) time, O(sessions) output; a gap of exactly the threshold stays in the session; reject a negative gap; a single event forms a one-event session.

<details>
<summary><b>Show solution and explanation</b></summary>

### 13a. Sort, then extend the open session

Sorting by `(user, ts, event_id)` makes the problem sequential: each event either extends the last open session for the same user when `when - last_end <= gap`, or opens a new one. The event id is the final tiebreak so equal timestamps produce a deterministic order.

The boundary is inclusive on purpose -- a gap of exactly `gap_seconds` is still the same session, which is how most product-analytics definitions are written. State the choice explicitly, because `<` versus `<=` changes session counts at the edge.

```python
from collections.abc import Iterable
from typing import Any


def sessionize(
    events: Iterable[dict[str, Any]],
    gap_seconds: int,
    user_key: str = "user",
    ts_key: str = "ts",
    id_key: str = "event_id",
) -> list[dict[str, Any]]:
    """Group events into sessions; a new session starts when the idle gap exceeds
    `gap_seconds`. The boundary is inclusive: a gap of exactly `gap_seconds`
    stays in the same session.
    """
    if gap_seconds < 0:
        raise ValueError("gap_seconds must be non-negative")
    ordered = sorted(events, key=lambda event: (event[user_key], event[ts_key], event[id_key]))
    sessions: list[dict[str, Any]] = []
    for event in ordered:
        user, when = event[user_key], event[ts_key]
        if (
            sessions
            and sessions[-1]["user"] == user
            and when - sessions[-1]["end"] <= gap_seconds
        ):
            session = sessions[-1]
            session["end"] = when
            session["count"] += 1
            session["duration"] = session["end"] - session["start"]
            session["event_ids"].append(event[id_key])
        else:
            sessions.append(
                {
                    "user": user,
                    "start": when,
                    "end": when,
                    "count": 1,
                    "duration": 0,
                    "event_ids": [event[id_key]],
                }
            )
    return sessions
```

</details>

**Complexity & edge cases.** O(n log n) if the input must be sorted, O(n) if the source already delivers ordered partitions per user; memory is O(sessions) for the output (or O(1) state per active user when streaming). Edge cases: out-of-order events need a lateness buffer before emitting; one event forms a one-event session; define `<=` vs `<` at the boundary (here exactly the gap is inactive); users with no events are absent, so downstream joins must be left joins; clock skew can make a user's timestamps non-monotonic.

**Interview follow-ups.**

1. *How would you sessionize an unbounded stream?* Keep per-user last-event state in a keyed store (Redis/Flink state) and emit a session when a timer for `last_event + gap` fires, handling late events with allowed lateness.
2. *How does this map to a Spark window function?* `LAG(ts)` over `PARTITION BY user ORDER BY ts`, then mark a new session when `ts - lag > gap`, and finally `SUM(new_session)` as a session id.

## Problem 14 - Production utilities: LRU cache and retry with backoff

**Statement.** Two small classes you will be asked to write (or reason about) in a data-platform interview: a fixed-capacity **LRU cache** with O(1) `get`/`put`, and a **retry** helper with capped exponential backoff plus optional jitter.

**Example.**

```text
capacity 2: put("a",1) put("b",2) get("a") -> 1; put("c",3) evicts "b"
backoff  : attempts=4, base=0.1, factor=2 -> sleeps [0.1, 0.2, 0.4]
```

**Constraints.** O(1) amortised `get`/`put`; validate capacity; injectable `sleep`/`rand` so the retry logic is testable and deterministic; cap both the delay and the attempt count; re-raise the final error.

<details>
<summary><b>Show solution and explanation</b></summary>

### 14a. `LRUCache` on top of `OrderedDict`

`OrderedDict` is a hash map plus a doubly linked list, which is exactly the two data structures an LRU cache needs. A hit calls `move_to_end` to mark the key most-recently-used; a write past capacity evicts with `popitem(last=False)`, the least-recently-used end. Both operations are O(1).

The hit/miss counters make the cache observable in tests. The documented pitfall: `get` returns `None` for a miss, so `None` cannot also be a stored value without a sentinel.

```python
import random
import time
from collections import OrderedDict
from collections.abc import Callable, Hashable
from typing import Generic, TypeVar

K = TypeVar("K", bound=Hashable)
V = TypeVar("V")


class LRUCache(Generic[K, V]):
    """O(1) get/put: an OrderedDict is a hash map plus a doubly linked list."""

    def __init__(self, capacity: int) -> None:
        if capacity <= 0:
            raise ValueError("capacity must be positive")
        self.capacity = capacity
        self._data: OrderedDict[K, V] = OrderedDict()
        self.hits = 0
        self.misses = 0

    def get(self, key: K) -> V | None:
        if key not in self._data:
            self.misses += 1
            return None
        self.hits += 1
        self._data.move_to_end(key)
        return self._data[key]

    def put(self, key: K, value: V) -> None:
        if key in self._data:
            self._data.move_to_end(key)
        self._data[key] = value
        if len(self._data) > self.capacity:
            self._data.popitem(last=False)

    def keys_oldest_first(self) -> list[K]:
        return list(self._data)
```

### 14b. One delay: `min(cap, base * factor**attempt)`

`compute_delay` is the pure core of the backoff policy: the attempt number goes in, the sleep before the next try comes out, capped by `max_delay` so a long outage does not create a 30-minute sleep. The random source is injected so a test can pin jitter to a known value.

Jitter interpolates between the deterministic delay (`jitter = 0`) and a random one (`jitter = 1`, "full jitter"), which is what desynchronises clients that failed together.

```python
def compute_delay(
    attempt: int,
    base: float,
    factor: float,
    max_delay: float,
    jitter: float,
    rand: Callable[[], float],
) -> float:
    """Delay before the retry after a zero-indexed failed `attempt`."""
    delay = min(max_delay, base * factor**attempt)
    if jitter:
        # Full jitter (jitter=1) spreads retries uniformly; partial jitter dampens.
        delay = delay * (1 - jitter) + delay * jitter * rand()
    return delay
```

### 14c. The whole schedule

`backoff_delays(attempts)` returns the sleep before each retry for a plan of `attempts` total tries, so there are `attempts - 1` sleeps: a plan of 4 tries sleeps 3 times. Expressing the schedule as a list makes it trivial to assert and to log as a plan before running it.

`attempts = 1` means "try once, never retry" and therefore an empty schedule.

```python
def backoff_delays(
    attempts: int,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    rand: Callable[[], float] = random.random,
) -> list[float]:
    """The sleep before each retry for a plan of `attempts` total tries."""
    return [compute_delay(n, base, factor, max_delay, jitter, rand) for n in range(attempts - 1)]
```

### 14d. The retry loop

Run the callable, catch only the declared exception types, and sleep before the next try; on the last attempt re-raise the original error instead of a wrapper, so the caller sees the real failure. `sleep` and `rand` are injected, which turns a timing-dependent routine into a deterministic one under test.

Retry only transient, idempotent operations: retrying a non-idempotent write double-writes. `attempts < 1` is rejected rather than treated as "no tries".

```python
def retry(
    func: Callable[[], V],
    attempts: int = 4,
    base: float = 0.1,
    factor: float = 2.0,
    max_delay: float = 10.0,
    jitter: float = 0.0,
    retry_on: tuple[type[BaseException], ...] = (Exception,),
    sleep: Callable[[float], None] = time.sleep,
    rand: Callable[[], float] = random.random,
) -> V:
    """Run `func`, retrying transient failures with capped exponential backoff."""
    if attempts < 1:
        raise ValueError("attempts must be >= 1")
    for attempt in range(attempts):
        try:
            return func()
        except retry_on:
            if attempt == attempts - 1:
                raise
            sleep(compute_delay(attempt, base, factor, max_delay, jitter, rand))
    raise RuntimeError("unreachable")
```

</details>

**Complexity & edge cases.** `OrderedDict` gives O(1) amortised `get`/`put` because it is a hash map plus a doubly linked list; `move_to_end` makes a hit the most recent and `popitem(last=False)` evicts the least recent. Edge cases: capacity zero/negative must be rejected, not silently no-op; `None` as a legal cached value is ambiguous with a miss, so use a sentinel; a hand-rolled version must update both the map and the list on every hit and eviction; retries must target idempotent, transient errors only; cap attempts and `max_delay` so a scheduler slot is not held forever; jitter is mandatory to avoid retry storms.

**Interview follow-ups.**

1. *Why does `functools.lru_cache` not solve every caching need?* It keys on hashable positional/keyword arguments, cannot expire by time, cannot be invalidated selectively, and is per-process rather than shared.
2. *What is full jitter and why does it work?* Sleep `uniform(0, min(cap, base * 2**n))`; randomising waits desynchronises clients that failed together, flattening the thundering herd when the service recovers.

## Beyond the standard library: pandas and PySpark

Interviewers often follow a pure-Python answer with "and how would you do this in pandas or Spark?". The
problems above deliberately use loops and dicts so the algorithm is visible; in production, pick the tool by the
shape of the work:

| Work | Tool |
| --- | --- |
| Streaming, branchy per-record logic, orchestration, API calls | plain Python (generators, dicts) |
| Column arithmetic, `groupby`, `merge`, rolling windows on data that fits in memory | pandas, Polars or DuckDB |
| Data that no longer fits one machine, or a shuffle that needs a cluster | PySpark |

Loops run per row in the interpreter; pandas runs vectorised C over whole columns, so a margin computed with a
loop over 200,000 rows is typically one to two orders of magnitude slower than the column expression:

```python
import pandas as pd

frame = pd.DataFrame({"revenue": [100.0, 0.0], "cost": [70.0, 5.0]})
slow = [(r.revenue - r.cost) / r.revenue if r.revenue else 0.0 for r in frame.itertuples()]
fast = ((frame["revenue"] - frame["cost"]) / frame["revenue"]).where(frame["revenue"] != 0, 0.0)
```

### Porting a pandas solution to PySpark

The port is mechanical once the grain is clear: `groupby` becomes `groupBy`, `merge` becomes `join`, and
`drop_duplicates(keep="last")` (Problem 12's latest-wins dedupe) becomes a `row_number()` window:

```python
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# pandas: orders.merge(customers, on="customer_id", how="left").groupby("country")["amount"].sum()
summary = (
    orders.join(customers, on="customer_id", how="left")
    .groupBy("country")
    .agg(F.count("*").alias("rows"), F.sum("amount").alias("revenue"))
)

# pandas: orders.sort_values("updated_at").drop_duplicates("order_id", keep="last")
latest = Window.partitionBy("order_id").orderBy(F.col("updated_at").desc())
deduped = orders.withColumn("_rank", F.row_number().over(latest)).filter("_rank = 1").drop("_rank")
```

Adaptation rules: never `toPandas()` a large frame (it collects everything to the driver); replace
`apply(axis=1)` with `F.when`/`F.expr`, and reach for `pandas_udf` only when no built-in exists, because Python
UDFs serialise rows and hide the logic from the optimiser. Keep the pure-Python or pandas version as the
correctness oracle and compare both on a small sample in tests. See [04 PySpark concepts](../../pyspark/pyspark-concepts-end-to-end.ipynb)
for runnable Spark examples.

## Interview questions

Twelve questions that come up when interviewers probe Python for data engineering, each with a concise model answer. Where a claim is checkable, a code cell in this section proves it.

### Q1. When do you reach for a generator instead of a list?

**Model answer.** A list materialises every result: O(n) memory and O(n) time up front. A generator produces one item per `next()` and holds O(1) state, so it can represent an unbounded or larger-than-RAM source. Use generators for file/stream ingestion and one-pass transform pipelines; use lists when you need random access, `len()`, reuse, or multiple passes.

### Q2. Why is `dict`/`set` lookup considered O(1), and what breaks it?

**Model answer.** They are hash tables: `hash(key)` picks a bucket, then equality confirms the match. Average-case lookup/insert/delete is O(1); iteration is O(n). Worst case is O(n) per operation under heavy collisions (a bad `__hash__` or adversarial keys). Keys must be hashable with value-based equality; a mutable key would break bucket invariants. Dicts preserve insertion order since Python 3.7, which is what makes an ordered dedupe a one-liner.

### Q3. `sorted()`, `list.sort()`, or `heapq.nlargest()` - which and why?

**Model answer.** `list.sort()` sorts in place, O(n log n) time, O(1) extra space (Timsort may use O(n/2) worst case), returns `None`. `sorted()` copies first, adding O(n) memory. `heapq.nlargest(k, it)` is O(n log k) time and O(k) memory, so it wins when k << n and preserves input order for ties. For top-N per group, keep one bounded heap per group.

### Q4. How do you deduplicate records while preserving order, and how does it scale?

**Model answer.** Stream the data and keep a `set` of seen keys, appending the first occurrence of each: O(n) time, O(k) memory for k distinct keys. At scale, partition by key so each partition dedupes independently; or sort by key and drop adjacent duplicates (spills to disk, O(n log n)); or use a Bloom filter as a cheap pre-filter, accepting false positives. In a warehouse: `ROW_NUMBER() OVER (PARTITION BY key ORDER BY ts DESC) = 1`.

### Q5. How do you process a file larger than RAM with only the standard library?

**Model answer.** Never call `read()`/`readlines()`. Iterate the file object line by line (bounded buffer), wrap it in a generator, process in fixed batches (`itertools.batched`), and keep only aggregates or the required state. Use `gzip.open(..., "rt")` for compressed files. Parallelise by byte ranges only when the format allows seeking to a record boundary; otherwise split by file. Write aggregates out incrementally.

### Q6. `json.load` vs `json.loads` vs streaming with `raw_decode`.

**Model answer.** `json.loads(str)` parses a string; `json.load(file)` reads a file object and parses it, but both materialise the whole document. For a JSON array too large for memory, drive `json.JSONDecoder().raw_decode(buffer, index)` to pull one value at a time, or use NDJSON where every line is an independent document. Third-party options are `ijson` and `jsonlines`.

### Q7. How does an in-memory hash join work, and when does it fail?

**Model answer.** Build a hash map on the smaller (dimension) side, then stream the larger (fact) side through it: O(n + m) time, O(m) memory for the build side. It fails when the build side does not fit in memory, when one join key is badly skewed, or when null/type semantics differ (`1` vs `"1"`, NULL never equals NULL). Remedies: broadcast the small side, sort-merge join, or hash-partition by key and join partition-wise.

### Q8. How do you handle late-arriving records and duplicates?

**Model answer.** Make the sink idempotent. Keep a map keyed by the business key and retain the record with the greatest `(event_timestamp, tiebreak)`, where the tiebreak is a Kafka offset or version so replays are deterministic. If producer clocks are unreliable, prefer the ingestion timestamp for lateness. Tombstones/deletes must also win, and the downstream write must merge/upsert rather than blindly append.

### Q9. What is schema drift and how do you detect it?

**Model answer.** Schema drift is any change in columns or types between batches. Infer the observed schema (column -> set of types) per batch, compare against the catalog or the previous batch, and classify added, removed, and retyped columns. Silent type changes such as int -> string are the dangerous ones. Diff declared Parquet/Avro metadata in O(columns); added required or removed fields break readers, while int -> float is usually benign widening.

### Q10. How do you compute percentiles and rolling windows efficiently?

**Model answer.** An exact percentile needs the sorted sample: sort once (O(n log n)) and each percentile is O(1). For rolling windows, means/sums are O(1) per element with a deque, but exact quantiles are not. Use an order-statistic tree (O(log n) insert) or an approximate sketch (t-digest, HDR histogram, Greenwald-Khanna) for large windows. Percentiles are not additive across buckets, and on a small sample p99 is just the maximum.

### Q11. How do you implement an LRU cache, and what does `functools.lru_cache` give you?

**Model answer.** Combine a hash map (O(1) lookup) with a doubly linked list or `OrderedDict` (recency order): `get` moves the key to the most-recent end, `put` inserts and evicts the least-recent end when over capacity. Both are O(1). `functools.lru_cache` is the C implementation for pure functions keyed on hashable arguments; it cannot key unhashable objects, has no time-based expiry, and caches per process. A sentinel is needed when `None` is a legal value.

### Q12. Why retry with exponential backoff and jitter?

**Model answer.** Delay before retry n is `min(cap, base * factor**n)`, plus jitter. Backoff gives a failing dependency time to recover; jitter (`uniform(0, delay)` for full jitter) desynchronises clients so they do not stampede the service the moment it returns. Retry only transient, idempotent operations (timeouts, 429, 503), cap attempts and max delay so no scheduler slot is held forever, and re-raise the final error after recording it.

## Python interview questions

Language questions that come up in the same interviews as the coding problems. Each has a short model answer; the two verification cells prove the checkable claims. The [Python lessons](../learning/python-01-values-and-types.ipynb) explain each topic in depth.

### Q13. What is the difference between a list and a tuple, and when do you prefer each?

**Model answer.** Lists are mutable and variable-length: use them for collections you grow or change, such as a batch of rows. Tuples are immutable, so a tuple of hashable values is hashable and can be a dict key or set member; use them for fixed-shape records such as `(customer_id, ds)` and for composite keys.

### Q14. Explain the mutable default argument trap.

**Model answer.** Defaults are evaluated once, when `def` runs, so `def f(item, bucket=[])` shares one list across every call: `f("a")` then `f("b")` returns `["a", "b"]`. Use `bucket=None` and create the list inside. Dataclasses have the same rule and require `field(default_factory=list)` for every mutable field.

### Q15. What is the difference between `__str__` and `__repr__`?

**Model answer.** `__repr__` is the unambiguous developer view, ideally re-evaluable like `Money(1250, 'USD')`; the REPL, debuggers and containers show it. `__str__` is the human view used by `print` and f-strings, and falls back to `__repr__`. If you write only one, write `__repr__`.

### Q16. Why must `__eq__` and `__hash__` be consistent?

**Model answer.** Dicts and sets find a key by hash, then confirm with `==`, so equal objects must have equal hashes or lookups miss keys that are present. Defining `__eq__` sets `__hash__` to `None`, which makes the class unhashable until you define `__hash__` over the same fields. `@dataclass(frozen=True)` generates both.

### Q17. When would you use `@classmethod` over `@staticmethod`?

**Model answer.** A classmethod receives `cls` and is the idiomatic alternative constructor (`Account.from_statement(...)`); because `cls` is the called class, it builds subclasses correctly. A staticmethod receives neither `self` nor `cls`: a namespaced pure helper. If it touches the class, use classmethod.

### Q18. What does a context manager guarantee?

**Model answer.** Teardown runs when the block exits, normally or by exception, so connections, locks, transactions and temporary files never leak. `__exit__` receives the exception; returning a truthy value swallows it, so transaction managers return `False` after a rollback. `contextlib.contextmanager` builds one from a generator with `try/finally` around the `yield`.

### Q19. What does `functools.wraps` do, and why does it matter?

**Model answer.** It copies `__name__`, `__doc__`, `__qualname__`, `__module__` and `__dict__` onto the wrapper and sets `__wrapped__`. Without it, logs, tracebacks and `inspect.signature` see the wrapper instead of the original function.

### Q20. Explain the GIL and its practical consequence.

**Model answer.** Only one thread executes Python bytecode at a time, so threads do not parallelise pure-Python CPU work. Blocking I/O releases the lock, so a `ThreadPoolExecutor` still speeds up many API or database calls. For CPU-bound work use `ProcessPoolExecutor`, a C extension that releases the GIL, or a distributed engine; asyncio is the single-threaded option for many concurrent network waits.

### Q21. What is the difference between `is` and `==`?

**Model answer.** `==` compares values through `__eq__`; `is` compares identity. Use `is` only for singletons such as `None`, `True`, `False` and sentinels. CPython caches small integers and interns some strings, so `is` can appear to work on them; that is an implementation detail and a latent bug.

### Q22. Why are type hints not enforcement?

**Model answer.** Annotations are stored in `__annotations__` and ignored at runtime, so `double("21")` runs even though the parameter is annotated `int`. Their value is static checking, editor support and documentation. At a trust boundary, validate at runtime with `isinstance`, a schema library or tests.

### Q23. How do you make an instance immutable?

**Model answer.** Use `@dataclass(frozen=True)`, which blocks assignment and generates `__hash__`, or `typing.NamedTuple`. Immutability makes values hashable and safe to share across threads. `frozen=True` is shallow: a list field inside a frozen dataclass can still be mutated.

### Q24. What is the difference between a shallow copy and a deep copy?

**Model answer.** A shallow copy (`list(x)`, `x.copy()`, `copy.copy`) creates a new outer container that shares the inner objects, so mutating a nested list shows through both. `copy.deepcopy` copies recursively. That is why `dict(config)` does not protect a nested `config["credentials"]`.

### Q25. `d[key]` versus `d.get(key)`, and when do `setdefault` or `defaultdict` help?

**Model answer.** `d[key]` raises `KeyError`, which is right when absence is a bug; `d.get(key, default)` suits optional settings. `setdefault(key, [])` and `defaultdict(list)` turn a group-by into a two-line loop, and `Counter` handles tallies. All rely on average O(1) lookup.

### Q26. Explain late binding in closures.

**Model answer.** A closure captures the variable, not its value, and looks it up when called. `[lambda: i for i in range(3)]` gives three callables that all return `2`. Bind the current value with `lambda i=i: i` or `functools.partial`.

### Q27. What does `for ... else` do?

**Model answer.** The `else` block runs only when the loop finished without `break`. It expresses "search, and handle not found" without a flag variable.

### Q28. Why prefer `Decimal` over `float` for money?

**Model answer.** Binary floats cannot represent most decimal fractions, so cents drift over many additions. `Decimal` does exact decimal arithmetic with explicit rounding. Build it from strings: `Decimal("19.99")`, because `Decimal(19.99)` inherits the binary error.

### Q29. What does `raise ... from` do?

**Model answer.** It sets the new exception's `__cause__`, so the traceback shows both errors with "the direct cause of" between them. You can raise an error that fits your API without losing the original. A bare `raise` in an `except` block re-raises the active exception unchanged.

### Q30. How does the LEGB rule work, and when do you need `global` or `nonlocal`?

**Model answer.** Lookup checks Local, Enclosing, Global, then Built-in scope. Reading an outer name needs no keyword; rebinding does (`nonlocal` for an enclosing function, `global` for a module name). Without it, `count += 1` makes `count` local and raises `UnboundLocalError`.